### Imports

In [ ]:
#!pip install -q vllm

import os
import cv2
import json
import glob
import math
import pydicom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#from vllm import LLM, SamplingParams
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score

## EDA
### Looking at the Studies
An initial look at the studies tells us there are 12 binary labels alongside an identifier and the full text report:
- StudyInstanceUID - unique identifier for the study; matches the folder name under train_series/.
- Report - the free-text radiology report. May be in any of several languages, depending on the reporting institution.
- ACL - anterior cruciate ligament injury (0/1).
- MCL - medial collateral ligament injury (0/1).
- Medial Meniscus - medial meniscus tear (0/1).
- Lateral Meniscus - lateral meniscus tear (0/1).
- Medial OA - osteoarthritis of the medial tibiofemoral compartment (0/1).
- Lateral OA - osteoarthritis of the lateral tibiofemoral compartment (0/1).
- PF OA - patellofemoral osteoarthritis (0/1).
- Effusion - joint effusion / excess fluid (0/1).
- Synovitis - inflammation of the joint lining (0/1).
- Baker's - Baker's cyst (0/1).
- Contusion - bone contusion / bone bruise (0/1).
- Fracture - fracture (0/1).

The report is in more languages than english so we're going to want to translate all of them to a single language. To avoid the model noticing the translation difference as well as the language difference, maybe we'll apply the translation technique to all reports even the english ones? Consider the approaches. We can see we have 4407 studies and a total of 24371 data points (MRI acquisitions). Most of our study data is null, the relavent information is actually stored in the reports. Exactly 58 studies have relavent information and all 58 of them retain all the information. So we have 58 complete studies and 4349 null studies where data is in the report. Within the 58 complete studies there's a pretty large skew to mostly negatives for some datapoints like MCL which will likely bias the data. When we extract the information from the reports we'll have to look at that again and perhaps cut to make them more even or randomly select a certain amount.

In [ ]:
# Setting up the base variables for later (base data url and label array)
BASE = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
TAGS = ['Rows', 'Columns', 'PixelSpacing', 'SliceThickness', 'SeriesDescription', 'Manufacturer']

train_info = pd.read_csv(f'{BASE}/train.csv')
train_info_series = pd.read_csv(f'{BASE}/train_series.csv')
sample_sub = pd.read_csv(f'{BASE}/sample_submission.csv')

# Want to view the general shape of our data
print("Shape train_info:", train_info.shape, "\nShape train_info Series:",train_info_series.shape, "\nShape Sample Submission:",sample_sub.shape)
train_info.head()

In [ ]:
# Want to see how much of the data is actually useful to work on since most of it seems to be null and get a decent idea of the spread.
pos = (train_info[LABELS] == 1).sum()
neg = (train_info[LABELS] == 0).sum()
pos_rate = pos / (pos + neg)
order = pos_rate.sort_values(ascending=False).index
pos, neg, pos_rate = pos[order], neg[order], pos_rate[order]
summary = pd.DataFrame({'non_null': train_info[LABELS].notna().sum(), 'positives': pos, 'negatives': neg})
summary['pos_rate'] = pos_rate
summary['pct_labeled'] = summary['non_null'] / len(train_info) * 100
summary = summary.sort_values('positives', ascending=False)

print(f"Studies with at least one non-null label: "
      f"{train_info[LABELS].notna().any(axis=1).sum()}\n")
print(summary.round(4))

print("\nLabels-per-study distribution:")
print(train_info[LABELS].notna().sum(axis=1).value_counts().sort_index())

In [ ]:
# Let's get some visuals in there to view the distribution as it exists of positive/negative non-nulls
fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(order))

ax.barh(y, pos, color='red', label='Positive (present)')
ax.barh(y, neg, left=pos, color='grey', label='Negative (absent)')

ax.set_yticks(y)
ax.set_yticklabels(order)
ax.invert_yaxis()
ax.set_xlabel('Number of studies')
ax.set_title('Label distribution across labeled studies')
ax.legend(loc='lower center')

# Annotate positive rate at the end of each bar
for i, (p, n) in enumerate(zip(pos, neg)):
    ax.text(p + n + 0.5, i, f'{pos_rate.iloc[i]:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

#### Looking at the Reports
Yeah going to need a language model to parse through all of that, maybe a basic RAG build as well to do it efficiently but two big things:
1. Multiple languages and no idea how much of which without manual or AI check
2. Multiple formats for how information including conclusions is shown so model will need a genuine understanding of the information
   - We probaly can't just do a basic scan through since there's nuance in the responses.

In [ ]:
# Get a solid look at the report distributions
print("Rows:", len(train_info))
print("Nulls in Report:", train_info['Report'].isna().sum())
print("Empty strings:", (train_info['Report'].fillna('').str.strip() == '').sum())

# Character stats
train_info['report_len'] = train_info['Report'].fillna('').str.len()
print("\nCharacter length:")
print(train_info['report_len'].describe().round(1))

# Word count
train_info['report_words'] = train_info['Report'].fillna('').str.split().str.len()
print("\nWord count:")
print(train_info['report_words'].describe().round(1))

In [ ]:
# Graph it cus why not
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_info['report_len'].clip(upper=train_info['report_len'].quantile(0.99)).hist(bins=50, ax=axes[0], edgecolor='black')
axes[0].set_title('Report length (characters, 99th pct clipped)')
axes[0].set_xlabel('chars')

train_info['report_words'].clip(upper=train_info['report_words'].quantile(0.99)).hist(bins=50, ax=axes[1], edgecolor='black', color='#d62728')
axes[1].set_title('Report word count (99th pct clipped)')
axes[1].set_xlabel('words')
plt.tight_layout()
plt.show()

In [ ]:
# Actually look at some of the reports
sample = train_info.sample(n=5, random_state=42)

for i, row in sample.iterrows():
    print("=" * 100)
    print(f"StudyInstanceUID: {row['StudyInstanceUID']}")
    print("-" * 100)
    print(row['Report'])
    print()

### Looking at the Series
An initial look at the series data tells us there are 2 binary labels alongside a study identifier, a series identifier and an image:
- StudyInstanceUID - study this series belongs to.
- SeriesInstanceUID - unique identifier for the series; matches the folder name under train_series/<StudyInstanceUID>/.
- Fluid_Sensitive - 1 if the sequence emphasizes fluid signal (T2, PD, STIR, and similar), 0 otherwise.
- Fat_Suppression - 1 if the sequence applies fat suppression, 0 otherwise. Note that although Fluid_Sensitive and Fat_Suppression are often correlated, as observed in the training set, they are not necessarily equivalent for every case.
- Anatomical_Plane - imaging plane: Sagittal, Coronal, or Axial.

The (1,0) and (0,1) combinations don't seem to exist.

In [ ]:
# Looking at an example series data point to get an understanding of it
study_id = train_info.iloc[0]['StudyInstanceUID']
series_dirs = glob.glob(f'{BASE}/train_series/{study_id}/*')
print(f"{len(series_dirs)} series for study {study_id}")
first_slice = sorted(glob.glob(f'{series_dirs[0]}/*.dcm'))[0]
ds = pydicom.dcmread(first_slice)
print("Modality: ", ds.Modality, "\nSeries Description: ", ds.SeriesDescription, "\nImage Shape: ", ds.pixel_array.shape)

In [ ]:
labeled_studies = train_info.dropna(subset=LABELS)['StudyInstanceUID'].tolist()
print(f"{len(labeled_studies)} labeled studies")

candidates = train_info_series[train_info_series['StudyInstanceUID'].isin(labeled_studies) & (train_info_series['Anatomical_Plane'] == 'Sagittal') &(train_info_series['Fluid_Sensitive'] == 1)]
print(f"{len(candidates)} matching series\n")

# Peek at a few, with the study's labels so we know what to look for
for _, row in candidates.head(5).iterrows():
    labels = train_info.loc[train_info['StudyInstanceUID'] == row['StudyInstanceUID'], LABELS].iloc[0]
    positives = [l for l in LABELS if labels[l] == 1]
    print("Study ID", row['StudyInstanceUID'], "\nSeries ID", row['SeriesInstanceUID'], "\nPositives:", positives, "\n")

print("\ntrain_series columns:", list(train_info_series.columns))
print("\ntrain_series dtypes:\n", train_info_series.dtypes)
print("\nNull counts per column:\n", train_info_series.isna().sum())

In [ ]:
# Get a solid distribution so we know what the spread looks like
print(train_info_series['Anatomical_Plane'].value_counts(dropna=False))
print("\nFluid_Sensitive:\n", train_info_series['Fluid_Sensitive'].value_counts(dropna=False))
print("\nFat_Suppression:\n", train_info_series['Fat_Suppression'].value_counts(dropna=False))

combo = train_info_series.groupby(['Anatomical_Plane', 'Fluid_Sensitive', 'Fat_Suppression']).size().unstack(fill_value=0)
print("\nPlane × Fluid × Fat combo:\n", combo)

ct = pd.crosstab(train_info_series['Anatomical_Plane'],[train_info_series['Fluid_Sensitive'], train_info_series['Fat_Suppression']])
ct.plot(kind='bar', stacked=True, figsize=(9, 5), edgecolor='black')
plt.ylabel('Number of series')
plt.title('Plane × (Fluid, Fat) combination')
plt.xticks(rotation=0)
plt.legend(title='(Fluid, Fat)', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Load some of the actual series data
def load_series(study_uid, series_uid, max_slices=None):
    path = f'{BASE}/train_series/{study_uid}/{series_uid}'
    slices = [pydicom.dcmread(f) for f in glob.glob(f'{path}/*.dcm')]

    orient = np.array(slices[0].ImageOrientationPatient, float)
    normal = np.cross(orient[:3], orient[3:])
    slices.sort(key=lambda s: np.dot(normal, np.array(s.ImagePositionPatient, float)))

    if max_slices and len(slices) > max_slices:
        idx = np.linspace(0, len(slices) - 1, max_slices).astype(int)
        slices = [slices[i] for i in idx]
    return slices

# Show the images relavent to that series
def show_montage(slices, ncols=4, title=None):
    n = len(slices)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
    axes = np.array(axes).reshape(-1)

    for i, s in enumerate(slices):
        img = s.pixel_array.astype(float)
        lo, hi = np.percentile(img, [1, 99])
        img = np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f"slice {i}", fontsize=8)
        axes[i].axis('off')

    for j in range(n, len(axes)):
        axes[j].axis('off')

    if title:
        fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
# Get a view of some of the actual images just for understanding honestly
study_uid = train_info_series['StudyInstanceUID'].iloc[0]
planes = ['Sagittal', 'Coronal', 'Axial']

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for r, plane in enumerate(planes):
    sub = train_info_series[(train_info_series['StudyInstanceUID'] == study_uid) & (train_info_series['Anatomical_Plane'] == plane)]
    s = load_series(sub.iloc[0]['StudyInstanceUID'], sub.iloc[0]['SeriesInstanceUID'], max_slices=6)
    for c in range(6):
        ax = axes[r, c]
        if c < len(s):
            img = s[c].pixel_array.astype(float)
            lo, hi = np.percentile(img, [1, 99])
            img = np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)
            ax.imshow(img, cmap='gray')
        ax.set_title(f"{plane} #{c+1}", fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Pull basic header info for every series
def series_info(study, series):
    path = f'{BASE}/train_series/{study}/{series}'
    files = sorted(f for f in os.listdir(path) if f.endswith('.dcm')) if os.path.isdir(path) else []
    if not files:
        return {'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'n_slices': 0}
    h = pydicom.dcmread(f'{path}/{files[0]}', stop_before_pixels=True, specific_tags=TAGS)
    return {'StudyInstanceUID': study, 'SeriesInstanceUID': series,
            'n_slices': len(files),
            'rows': h.get('Rows'), 'cols': h.get('Columns'),
            'pixel_spacing': float(h.PixelSpacing[0]) if 'PixelSpacing' in h else None,
            'thickness': float(h.SliceThickness) if 'SliceThickness' in h else None,
            'description': h.get('SeriesDescription', ''),
            'manufacturer': h.get('Manufacturer', ''),
            'middle_file': f'{path}/{files[len(files) // 2]}'}

rows = Parallel(n_jobs=16, prefer='threads')(delayed(series_info)(st, se) for st, se in zip(train_info_series['StudyInstanceUID'], train_info_series['SeriesInstanceUID']))
series_meta = train_info_series.merge(pd.DataFrame(rows), on=['StudyInstanceUID', 'SeriesInstanceUID'])

# Does the CSV match the real data?
print("Series with no files:", (series_meta['n_slices'] == 0).sum())
print("Duplicate rows in train_series.csv:", train_info_series.duplicated().sum())
print("Series IDs used in more than one study:", (train_info_series.groupby('SeriesInstanceUID')['StudyInstanceUID'].nunique() > 1).sum())
print("Studies with no series:", (~train_info['StudyInstanceUID'].isin(train_info_series['StudyInstanceUID'])).sum())

# Duplicate slices inside a series: needs every header, so only check a sample
def n_unique_positions(study, series):
    path = f'{BASE}/train_series/{study}/{series}'
    pos = [pydicom.dcmread(f'{path}/{f}', stop_before_pixels=True, specific_tags=['ImagePositionPatient']).get('ImagePositionPatient') for f in os.listdir(path) if f.endswith('.dcm')]
    return len({tuple(p) for p in pos if p is not None})

sample = series_meta[series_meta['n_slices'] > 0].sample(300, random_state=42)
sample['n_unique_positions'] = Parallel(n_jobs=16, prefer='threads')(delayed(n_unique_positions)(st, se) for st, se in zip(sample['StudyInstanceUID'], sample['SeriesInstanceUID']))
dup_slices = sample[sample['n_unique_positions'] < sample['n_slices']]
print("\nSampled series with repeated slice positions:", len(dup_slices), "/ 300")
dup_slices[['StudyInstanceUID', 'SeriesInstanceUID', 'description', 'n_slices', 'n_unique_positions']].head()

In [ ]:
# Let's get a nice idea of the spread
print(series_meta['manufacturer'].value_counts())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
series_meta['n_slices'].hist(bins=50, ax=axes[0], edgecolor='black'); axes[0].set_title('Slices per series')
series_meta['rows'].hist(bins=50, ax=axes[1], edgecolor='black'); axes[1].set_title('Image rows (pixels)')
series_meta['pixel_spacing'].hist(bins=50, ax=axes[2], edgecolor='black'); axes[2].set_title('Pixel spacing (mm)')
plt.tight_layout()
plt.show()

### Processing Images
Let's get a quick processing version for others to work on but from going over this the processing is relatively simple. Duplicate removal, some level of grouping though depends on how the model is reading things.

In [ ]:
# Process the fluid-sensitive series for all 3 planes (sagittal, coronal, axial)
IMG_SIZE = 224
N_SLICES = 24
RUN_ALL = True   # False = 50 knees per plane (quick test). True for the committed run.

def process_series(study, series, out_dir):
    try:
        vol = np.stack([cv2.resize(s.pixel_array.astype(np.float32), (IMG_SIZE, IMG_SIZE),
                                   interpolation=cv2.INTER_AREA)
                        for s in load_series(study, series)])
        vol = vol[np.linspace(0, len(vol) - 1, N_SLICES).round().astype(int)]
        lo, hi = np.percentile(vol, [1, 99])
        vol = np.clip((vol - lo) / (hi - lo + 1e-8), 0, 1)
        np.save(f'{out_dir}/{study}.npy', (vol * 255).astype(np.uint8))
        return ''
    except Exception as e:
        return str(e)

labels = pd.read_csv('/kaggle/input/datasets/alextanasescu/training-labels/Training Labels.csv')

for PLANE in ['Sagittal', 'Coronal', 'Axial']:
    OUT_DIR = f'/kaggle/working/images_{PLANE.lower()}'
    os.makedirs(OUT_DIR, exist_ok=True)

    # Fluid-sensitive series in this plane; if a study has more than one, keep the one with the most slices
    chosen = (series_meta[(series_meta['Anatomical_Plane'] == PLANE) & (series_meta['Fluid_Sensitive'] == 1)]
              .sort_values('n_slices', ascending=False)
              .drop_duplicates('StudyInstanceUID'))
    print(f"\n{PLANE}: {len(chosen)} / {len(train_info)} studies have a fluid-sensitive series")

    todo = chosen if RUN_ALL else chosen.head(50)
    errors = Parallel(n_jobs=-1, verbose=0)(
        delayed(process_series)(st, se, OUT_DIR)
        for st, se in zip(todo['StudyInstanceUID'], todo['SeriesInstanceUID']))
    todo = todo.assign(error=errors)
    print("Failed:", (todo['error'] != '').sum())

    # Manifest: one row per study with its image file, labels and train/val split
    done = todo.loc[todo['error'] == '', ['StudyInstanceUID', 'SeriesInstanceUID']]
    manifest = labels.merge(done, on='StudyInstanceUID')
    manifest['path'] = OUT_DIR + '/' + manifest['StudyInstanceUID'] + '.npy'
    manifest = manifest[~manifest['duplicate_of_labeled']]                 # keep the duplicate out
    manifest['split'] = np.where(manifest['is_labeled'], 'val', 'train')   # real labels held out for val
    manifest.to_csv(f'/kaggle/working/manifest_{PLANE.lower()}.csv', index=False)
    print(manifest['split'].value_counts().to_dict())


In [ ]:
# Quick look at a few processed volumes with their labels
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
for r, (_, row) in enumerate(manifest.sample(3, random_state=0).iterrows()):
    vol = np.load(row['path'])
    for c, i in enumerate(np.linspace(0, N_SLICES - 1, 6).astype(int)):
        axes[r, c].imshow(vol[i], cmap='gray')
        axes[r, c].axis('off')
    axes[r, 0].set_title(', '.join(l for l in LABELS if row[l] > 0.5) or 'no positives', fontsize=8, loc='left')
plt.tight_layout()
plt.show()